In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings("ignore") 

In [ ]:

df = pd.read_csv(
    "D:\\e drive\\project\\netflix recommendation system\\src\\data\\movies_metadata.csv"
)
df1 = pd.read_csv(
    "D:\\e drive\\project\\netflix recommendation system\\src\\data\\ratings_small.csv"
)
df2 = pd.read_csv(
    "D:\\e drive\\project\\netflix recommendation system\\src\\data\\credits.csv"
)
df3 = pd.read_csv(
    "D:\\e drive\\project\\netflix recommendation system\\src\\data\\link.csv"
)

df['id'] = pd.to_numeric(df['id'], errors='coerce')



In [ ]:
df.columns

In [ ]:
df1.columns

In [ ]:
dataframe = pd.merge(
    df,
    df2,
    on='id',
    how='left'
)


In [ ]:
dataframe.head()
dataframe.columns

In [ ]:
moviesrating =dataframe[['id','title','vote_average','vote_count']]

In [ ]:
moviesrating.head()

In [ ]:
moviesrating.shape

In [ ]:
moviesrating.isnull().sum()

In [ ]:
moviesrating.dropna(inplace=True)

In [ ]:
moviesrating.isnull().sum()

In [ ]:
moviesrating.shape
moviesrating.info()

In [ ]:
moviesrating.loc[8031]


RATING BASED RECCOMANDATION SYSTEM!!!!!

m= minimum votes required for rating in top movies
c= mean of avg vote count


In [ ]:
C= moviesrating['vote_average'].mean()
m= moviesrating['vote_count'].quantile(0.99)
filtered = moviesrating[moviesrating['vote_count'] > m]
filtered.sort_values('vote_count',ascending=False)



In [ ]:
from src.utils import weighted_ratings

In [ ]:
filtered['score'] = filtered.apply(lambda x: weighted_ratings(x, m, C), axis=1)


In [ ]:
filtered.sort_values('score',ascending=False).head(10)

ratings recommandation done!!!

trending movies


In [ ]:
dataframe.columns

In [ ]:
dataframe['popularity'] = pd.to_numeric(dataframe['popularity'], errors='coerce')


In [ ]:
dataframe['popularity']=dataframe['popularity'].fillna(0).astype(int)

In [ ]:
dff=dataframe.sort_values(by='popularity' ,ascending=False).head(10)

In [ ]:
plt.Figure(figsize=(4,5))
plt.barh(dff['title'],dff['popularity'], color="RED")
plt.ylabel('movies')
plt.xlabel('trending')
plt.title("Trending Movies by Popularity")
plt.gca().invert_yaxis()
plt.show()

content based filtering !!! 

In [ ]:
dataframe.columns

In [ ]:
data = dataframe[['title','genres', 'crew', 'cast','overview']]

In [ ]:
data.head()

In [ ]:
data.info()
data.isnull().sum()

In [ ]:
print(data.loc[data['title'] == "Jumanji", 'crew'])


In [ ]:
def clean_genres(obj):
    try:
        obj=ast.literal_eval(obj)
        names=[d['name'] for d in obj][:5]
    except:
        names=[]
    return names    

data['genres']=data['genres'].apply(clean_genres)

In [ ]:
import ast
def clean_cast(obj):
    try:
        obj=ast.literal_eval(obj)
        names=[d['name'] for d in obj][:5]
    except:
        names=[]
    return names    

data['cast']=data['cast'].apply(clean_cast)


In [ ]:
data['cast']

In [ ]:


def clean_crew(obj):
    try:
        # If string, parse into Python list
        if isinstance(obj, str):
            obj = ast.literal_eval(obj)
        
        # Ensure it's iterable
        if isinstance(obj, list):
            directors = [d.get('name') for d in obj if d.get('job','').lower() == 'director']
        else:
            directors = []
    except Exception as e:
        print("Error:", e, "with obj:", obj)  # debug
        directors = []
    return directors

data['crew'] = data['crew'].apply(clean_crew)


In [ ]:
data['crew']

In [ ]:
data['crew'].isnull().sum()

In [ ]:
data['cast'] = data['cast'].apply(lambda x: " ".join(x))
data['crew'] = data['crew'].apply(lambda x: " ".join(x))
data['genres'] = data['genres'].apply(lambda x: " ".join(x))


In [ ]:
features = ['genres', 'overview', 'cast', 'crew']
for feature in features:
    data[feature] = data[feature].fillna('')


In [ ]:
data = data.dropna(subset=['title'])


In [ ]:
data.isnull().sum()

In [ ]:
data.head()

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity 

data['soup'] = data.apply(
    lambda x: "".join(x['genres']) + " " +
              "".join(x['crew'])   + " " +
              "".join(x['cast'])   + " " +
              x['overview'],
    axis=1
)



In [ ]:
data['soup'].head()
data['soup'].str.lower()


In [ ]:
cv=CountVectorizer(max_features=5000,stop_words='english')
vectors= cv.fit_transform(data['soup'])



In [ ]:
def movies_recommendation(movie):
    # normalize input
    movie = movie.lower()

    # check if movie exists
    if movie not in data['title'].str.lower().values:
        print('Movie not found!')
        return []

    # find index of the movie
    index = data[data['title'].str.lower() == movie].index[0]

    # similarity scores for that movie
    distance = cosine_similarity(vectors[index], vectors).flatten() 

    movie_list = sorted(list(enumerate(distance)), key=lambda x: x[1], reverse=True)[1:10]

    return [data.iloc[i[0]].title for i in movie_list]


In [ ]:

print(movies_recommendation("Avatar"))


In [ ]:
print(movies_recommendation('JUMANJI'))

collaborative filtering!!!!!!

In [ ]:
data2 = pd.merge(df1, df3, on='movieId', how='left')

# fix imdbId (remove decimals properly)
data2['imdbId'] = data2['imdbId'].astype('Int64').astype(str).str.zfill(7)
data2['imdbId'] = 'tt' + data2['imdbId']

# merge with metadata
data2 = pd.merge(
    data2,
    df[['imdb_id', 'title']],
    left_on='imdbId',
    right_on='imdb_id',
    how='left'
)

# drop extras
data2 = data2.drop(columns=['imdb_id', 'tmdbId'])

print(data2.head(10))
print(data2[['movieId','imdbId','title']].dropna().head(10))


In [ ]:
data2.isnull().sum()

In [ ]:
user_matrix= data2.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
)

In [ ]:
user_matrix

user based similarity

In [ ]:
matrix = user_matrix.fillna(0)
user_similarity= cosine_similarity(matrix)
user_similarity_df =pd.DataFrame(
    user_similarity,
    index= matrix.index,
    columns =matrix.index
)


In [ ]:
def user_based_recommend(user_id, matrix, user_similarity_df, top_n=10):
    # ratings of the target user
    user_ratings = matrix.loc[user_id]
    
    # find users similar to the target
    similar_users = user_similarity_df[user_id].sort_values(ascending=False)[1:6]  
    # top 5 similar users (excluding self)

    scores = {}
    for other_user, sim in similar_users.items():
        # ratings of this similar user
        other_ratings = matrix.loc[other_user]
        
        # for each movie this user rated but target has not watched
        for movie, rating in other_ratings[other_ratings > 0].items():
            if user_ratings[movie] == 0:  # target hasn't watched
                # accumulate score weighted by similarity
                scores[movie] = scores.get(movie, 0) + sim * rating

    # sort movies by score
    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]
    id=[movie for movie, _ in sorted_scores]
    recommended_titles = data2[data2['movieId'].isin(id)]['title'].unique().tolist()
    return recommended_titles

In [ ]:
print(user_based_recommend(1, matrix, user_similarity_df, top_n=10))



item based similarity

In [ ]:
item_similarity = cosine_similarity(matrix.T)
item_similarity_df = pd.DataFrame(item_similarity, 
                                  index=matrix.columns, 
                                  columns=matrix.columns)


In [ ]:
def item_specific_recommendation(user_id, item_similarity_df, top_n=10):
    # get ratings of the given user
    user_ratings = matrix.loc[user_id]

    # movies not watched yet
    unwatched = user_ratings[user_ratings == 0].index
    
    scores = {}
    for movie in unwatched:
        sim_movie = item_similarity_df[movie]

        # weighted average score
        score = (sim_movie * user_ratings).sum() / sim_movie.sum()

        scores[movie] = score

    # top N recommendations
    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]
    ids= [movie for movie, _ in sorted_scores]
    recommended_titles = data2[data2['movieId'].isin(ids)]['title'].tolist()
    return recommended_titles

print(item_specific_recommendation(1, item_similarity_df))


In [ ]:
print(item_specific_recommendation(4, item_similarity_df))

